# 模組 3 · 光譜前處理
## 散射校正、特徵增強、樣本增強、訊號轉換、正交化、小波去噪

掌握 NIRS 的核心 — 前處理。學會四大類前處理、自動探索組合、資料擴增與進階去噪。

**對應官方範例**：`examples/user/03_preprocessing/`（U01–U06）

## 🚀 環境設定（請先執行下面這格）

本課程使用 **nirs4all 官方範例資料集（sample_data）**。下面這格會：
1. 從 GitHub 下載 nirs4all（內含 0.9 版函式庫與官方光譜資料集）
2. 安裝函式庫
3. 切換到 `examples/` 目錄（裡面有 `sample_data/`）

> 💡 **為什麼從 GitHub 安裝？** PyPI（`pip install nirs4all`）目前是舊版 0.4，沒有本課程使用的 `nirs4all.run()` 簡易 API。GitHub 上的版本（0.9+）才有，且需要 **Python ≥ 3.11**（Colab 預設 3.12，沒問題）。

第一次執行約需 1–2 分鐘，請耐心等候出現「✅ 完成」。

In [ ]:
# === Colab / Jupyter 環境設定（每本 notebook 開頭執行一次）===
import os, sys, subprocess

if not os.path.isdir('nirs4all'):
    print('⬇️  下載 nirs4all 與官方資料集（約 1–2 分鐘）…')
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/GBeurier/nirs4all.git'], check=True)
    print('📦 安裝 nirs4all …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    './nirs4all'], check=True)

# 切換到 examples 目錄（內含 sample_data）
if os.path.basename(os.getcwd()) != 'examples' and os.path.isdir('nirs4all/examples'):
    os.chdir('nirs4all/examples')

import nirs4all
print('✅ 完成！nirs4all 版本：', nirs4all.__version__)
print('   工作目錄：', os.getcwd())
print('   可用資料集：', [d for d in os.listdir('sample_data') if os.path.isdir(os.path.join('sample_data', d))])

---
## U01 · 前處理基礎：NIRS 專屬轉換  ★★☆☆☆

NIRS 前處理四大類：**散射校正**（SNV、MSC）、**基線校正**（Detrend）、**導數**（一階去常數、二階去線性基線）、**平滑**（Gaussian、Savitzky-Golay）。導數凸顯峰但放大雜訊，常與平滑併用。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import (
    StandardNormalVariate, MultiplicativeScatterCorrection, Detrend,
    FirstDerivative, SavitzkyGolay)

# 比較幾種前處理的效果
for name, prep in [("無", None),
                   ("SNV", StandardNormalVariate()),
                   ("一階導數", FirstDerivative()),
                   ("SNV+一階導數", None)]:
    steps = [MinMaxScaler(), {"y_processing": MinMaxScaler()}]
    if name == "SNV+一階導數":
        steps += [StandardNormalVariate(), FirstDerivative()]
    elif prep is not None:
        steps.append(prep)
    steps += [ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
              {"model": PLSRegression(n_components=10)}]
    r = nirs4all.run(pipeline=steps, dataset="sample_data/regression",
                     name=name, verbose=0)
    print(f"{name:14s} → RMSE {r.best_rmse:.4f}, R² {r.best_r2:.4f}")

> ✍️ **練習**：再加入 `Detrend()` 與 `MultiplicativeScatterCorrection()` 進比較迴圈，找出最佳前處理。

---
## U02 · 特徵增強：自動探索前處理  ★★★☆☆

`feature_augmentation` 自動建立前處理搜尋空間。三種 `action` 模式：
- **extend**：獨立新增選項（平行探索）
- **add**：在既有基礎上串接、保留原始（消融研究）
- **replace**：串接後丟棄原始（多階段乾淨鏈）

最佳前處理鏈在 `result.best['preprocessings']`，格式如 `"SNV|FirstDerivative"`。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate, FirstDerivative, SavitzkyGolay

pipeline = [
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    {"feature_augmentation": [StandardNormalVariate, FirstDerivative, SavitzkyGolay],
     "action": "extend"},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="FeatAug", verbose=1)

print("\n最佳前處理鏈：", result.best.get('preprocessings'))
for p in result.top(n=5, display_metrics=['rmse', 'r2']):
    print(f"  {p.get('preprocessings','-'):25s} RMSE={p['rmse']:.4f}")

> ✍️ **練習**：把 `action` 改成 `"add"` 做消融研究，觀察串接前處理是否逐步改善。

---
## U03 · 樣本增強：資料擴增  ★★★☆☆

`sample_augmentation` 對訓練資料做擴增（加雜訊、基線漂移、波長平移），增加多樣性與韌性。`count` 控制每個原始樣本產生幾個增強樣本；`balance="y"` 可平衡類別/區間。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.augmentation import Rotate_Translate, GaussianAdditiveNoise

pipeline = [
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    {"sample_augmentation": {
        "transformers": [Rotate_Translate, GaussianAdditiveNoise(sigma=0.01)],
        "count": 2,                 # 每個原始樣本產生 2 個
        "selection": "random",
        "random_state": 42,
    }},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="SampleAug", verbose=1)
print("RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：把 `count` 從 2 加到 4，比較有無擴增的測試 RMSE，評估擴增效益。

---
## U04 · 訊號轉換：吸光度 / 反射率 / Kubelka-Munk  ★★★☆☆

不同儀器輸出不同訊號型態，建模前常統一到吸光度。`ToAbsorbance`（R→A）、`FromAbsorbance`（A→R）；粉末/顆粒樣品常用 `KubelkaMunk`。所有轉換器相容 sklearn。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import StandardNormalVariate

# 示範：把訊號轉換放進 pipeline（這裡用 SNV 代表前處理位置）
# 若你的資料是反射率，可改用 ToAbsorbance(source_type="reflectance")
pipeline = [
    StandardNormalVariate(),
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="SignalConv", verbose=1)
print("RMSE:", round(result.best_rmse, 4))

# 自動偵測訊號型態
from nirs4all.data.signal_type import detect_signal_type
import numpy as np
print("（範例）偵測函式：detect_signal_type(X) → (型態, 信心, 原因)")

> ✍️ **練習**：若你的真實資料是反射率%，試著在最前面串 `PercentToFraction()` + `ToAbsorbance()` 再建模。

---
## U05 · 正交化：OSC 與 EPO  ★★★☆☆

進階前處理。**OSC** 移除與目標 Y 正交（無用）的變異；**EPO** 移除與外部參數（溫度、批次）相關的變異。兩者讓模型聚焦在真正相關的光譜變異，提升可解釋性。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import OSC

# OSC：去除與目標正交的變異
pipeline = [
    OSC(),
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="OSC", verbose=1)
print("OSC 後 RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：比較有/無 OSC 的 RMSE。在有批次效應的資料（M2-U06）上，EPO 通常更有感。

---
## U06 · 小波去噪：WaveletDenoise  ★★★☆☆

`WaveletDenoise` 用多階小波分解，對細節係數做門檻處理後重建，能保留光譜形狀同時去除高頻雜訊。參數：`wavelet`（db4、sym8…）、`level`、`threshold_mode`（soft 平滑 / hard 保峰）。

In [ ]:
import nirs4all
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import ShuffleSplit
from sklearn.cross_decomposition import PLSRegression
from nirs4all.operators.transforms import WaveletDenoise

pipeline = [
    WaveletDenoise(wavelet='db4', level=3, threshold_mode='soft',
                   noise_estimator='median'),
    MinMaxScaler(), {"y_processing": MinMaxScaler()},
    ShuffleSplit(n_splits=3, test_size=0.25, random_state=42),
    {"model": PLSRegression(n_components=10)},
]
result = nirs4all.run(pipeline=pipeline, dataset="sample_data/regression",
                      name="Wavelet", verbose=1)
print("小波去噪後 RMSE:", round(result.best_rmse, 4))

> ✍️ **練習**：比較 `threshold_mode='soft'` 與 `'hard'` 對結果的影響。